In [10]:
# COLAB SETUP

!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 72 (delta 16), reused 62 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 861.85 KiB | 4.90 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/proto-tsrl


In [15]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.experiments.ppg.ppg_model import PPGModel
import src.utils.sampling_utils as sampling_utils
import src.utils.training_utils as training_utils

In [ ]:
# SETTINGS

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# data
INCLUDE_CLEAN_DATA = True
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = True
DATASET = 'train'

# architecture
MODEL = PPGModel(representation_dimension = 10)
if torch.cuda.device_count() > 1:
    MODEL = nn.DataParallel(MODEL)
elif not isinstance(MODEL, nn.DataParallel):
    MODEL = nn.DataParallel(MODEL)

# schedule
N_EPOCHS_STAGE1 = 6
N_EPOCHS_STAGE2 = 6
N_EPOCHS_STAGE3 = 3

# logging
SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok = True)
FNAME = f'prototsrl_ppg'

# optimizer
OPTIMIZER = {}

SCHEDULER = {}

In [14]:
# DATA

# quality: high - [0,0,1] med - [0,1,0] low - [1,0,0]
# afib: pos - [0,1]
def load_data(clean, seminoisy, noisy, dataset):
    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/ppg_data/'
    with np.load(file_path + f'{dataset}.npz') as data:
        ppg_signal = data['signal']
        ppg_qual = data['qa_label']
        rhythm = data['rhythm']

    qual_mask = np.zeros(ppg_qual.shape[0], dtype = bool)
    if clean:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 0, 1]), axis = 1))
    if seminoisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 1, 0]), axis = 1))
    if noisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([1, 0, 0]), axis = 1))

    X, y = ppg_signal[qual_mask], rhythm[qual_mask]
    X = np.transpose(X, (0, 2, 1))

    afib_label = np.array([0, 1])
    y = np.all(y == afib_label, axis = 1)

    return X, y

X_train, y_train = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = DATASET
)

print(f'X shape: {X_train.shape}')
print(f'y shape: {y_train.shape}')

class PPGDataset(Dataset):

    def __init__(self, signals, labels):
        self.signals = signals
        self.labels = labels

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        x = self.signals[idx]
        y = self.labels[idx]
        return x, y

def make_dataloaders(X, y, batch_size = 256, val_ratio = 0.01, seed = 42, num_workers = 4):
    sss = StratifiedShuffleSplit(n_splits = 1, test_size = val_ratio, random_state = seed)
    train_idx, val_idx = next(sss.split(X, y))

    sig_train = X[train_idx].astype(np.float32)
    y_train = y[train_idx].astype(np.float32)
    sig_val = X[val_idx].astype(np.float32)
    y_val = y[val_idx].astype(np.float32)

    ds_train = PPGDataset(sig_train, y_train)
    ds_val = PPGDataset(sig_val, y_val)

    dl_train = DataLoader(
        ds_train,
        batch_size = batch_size,
        shuffle = True,
        num_workers = num_workers,
        pin_memory = True)
    dl_val = DataLoader(
        ds_val,
        batch_size = batch_size,
        shuffle = False,
        num_workers = num_workers,
        pin_memory = True)

    return dl_train, dl_val

dl_train, dl_val = make_dataloaders(X_train, y_train)